# File Prober

This notebook probes unprocessed video files in the database, queries media information using ffprobe, and displays results.

Workflow:
1. Query all files that exist in the database but have no encode record yet
2. Run ffprobe on each file to extract metadata (resolution, codec, duration, etc.)
3. Display results and handle errors gracefully

In [2]:
import sqlite3
import subprocess
import json
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


## Define File Probing Functions

In [9]:
def get_unprocessed_files(db_path='boilest.db'):
    """
    Get all files that exist in the files table but don't have an encode record yet.
    
    Args:
        db_path (str): Path to the database file
    
    Returns:
        list: List of unprocessed files with guid, file_path, file_name
    """
    try:
        conn = sqlite3.connect(str(db_path))
        conn.row_factory = sqlite3.Row
        cur = conn.cursor()
        
        # LEFT ANTI JOIN: files that have no corresponding encode record
        cur.execute("""
            SELECT f.guid, f.file_path, f.file_name, f.directory_guid
            FROM files f
            LEFT JOIN encode e ON f.guid = e.guid
            WHERE e.guid IS NULL
        """)
        
        unprocessed = [dict(row) for row in cur.fetchall()]
        conn.close()
        
        return unprocessed
    
    except Exception as e:
        print(f"✗ Error querying database: {e}")
        return []

In [19]:
def run_ffprobe(file_path):
    """
    Run ffprobe on a file and return media information.
    
    Args:
        file_path (str): Full path to the media file
    
    Returns:
        dict: ffprobe output (streams, format info) or error details
    """
    try:
        cmd = [
            'ffprobe',
            '-loglevel', 'quiet',
            '-show_entries', 'format:stream=index,stream,codec_type,codec_name,channel_layout,format=nb_streams',  

                       

            '-of', 'json',
            file_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode != 0:
            return {'error': f'ffprobe failed: {result.stderr}'}
        
        probe_data = json.loads(result.stdout)
        
        # Pretty-print the ffprobe JSON output
        print(json.dumps(probe_data, indent=2))
        
        return probe_data
    
    except FileNotFoundError:
        return {'error': 'ffprobe not found. Ensure ffmpeg is installed and in PATH.'}
    except subprocess.TimeoutExpired:
        return {'error': 'ffprobe timeout (file too large or network issue)'}
    except json.JSONDecodeError:
        return {'error': 'Invalid ffprobe JSON output'}
    except Exception as e:
        return {'error': str(e)}


In [18]:
output = run_ffprobe('\media/Media 1/test_file_01.mp4')

{
  "programs": [],
  "stream_groups": [],
  "streams": [
    {
      "index": 0,
      "codec_name": "h264",
      "codec_type": "video"
    }
  ],
  "format": {
    "filename": "\\media/Media 1/test_file_01.mp4",
    "nb_streams": 1,
    "nb_programs": 0,
    "nb_stream_groups": 0,
    "format_name": "mov,mp4,m4a,3gp,3g2,mj2",
    "format_long_name": "QuickTime / MOV",
    "start_time": "0.000000",
    "duration": "4.254250",
    "size": "775034",
    "bit_rate": "1457430",
    "probe_score": 100,
    "tags": {
      "major_brand": "isom",
      "minor_version": "512",
      "compatible_brands": "isomiso2avc1mp41",
      "encoder": "Lavf58.76.100"
    }
  }
}


In [ ]:








def check_codecs(stream_info):
    # Loops through the streams in stream_info from requires_encoding, then
    # calls functions to determine if the steam needs encoding based on stream type conditions 
    streams_count = stream_info['format']['nb_streams']
    encoding_decision = 'False'
    ffmpeg_command = str()
    print ('There are : ' + str(streams_count) + ' streams')
    for i in range (0,streams_count):
        codec_type = stream_info['streams'][i]['codec_type'] 
        if codec_type == 'video':
            print('Stream ' + str(i) + ' is video')
            encoding_decision, ffmpeg_command = check_video_stream(encoding_decision, i, stream_info, ffmpeg_command)
        elif codec_type == 'audio':
            encoding_decision, ffmpeg_command = check_audio_stream(encoding_decision, i, stream_info, ffmpeg_command)
            print('audio stream')
        elif codec_type == 'subtitle':
            encoding_decision, ffmpeg_command = check_subtitle_stream(encoding_decision, i, stream_info, ffmpeg_command)
            print('subtitle stream')
        elif codec_type == 'attachment':
            encoding_decision, ffmpeg_command = check_attachmeent_stream(encoding_decision, i, stream_info, ffmpeg_command) 
            print('attachment stream')    
    print (encoding_decision)   
    print (ffmpeg_command)
   # return encoding_decision, ffmpeg_command

def check_video_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the video stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    desired_video_codec = 'av1'
    print('Steam ' + str(i) + ' codec is: ' + codec_name)
    if codec_name == desired_video_codec:
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v copy'
    elif codec_name == 'mjpeg':
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v copy'
    elif codec_name != desired_video_codec: 
        encoding_decision = True
        svt_av1_string = "libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15"
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v ' + svt_av1_string
    else:
        print('ignoring for now')
    return encoding_decision, ffmpeg_command


def check_audio_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the audio stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    # This will be populated at a later date
    #desired_audio_codec = 'aac'
    #if codec_name != desired_video_codec:
    #    encoding_decision = True
    print('Steam ' + str(i) + ' codec is: ' + codec_name)
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:a copy'
    return encoding_decision, ffmpeg_command


def check_subtitle_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the subtitle stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    # This will be populated at a later date
    #desired_subtitle_codec = 'srt'
    #if codec_name != desired_subtitle_codec:
    #    encoding_decision = True
    print('Steam ' + str(i) + ' codec is: ' + codec_name)
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:s copy'
    return encoding_decision, ffmpeg_command


def check_attachmeent_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the attachment stream from check_codecs to determine if the stream needs encoding
    # This will be populated at a later date
    #desired_attachment_codec = '???'
    #if codec_name != desired_attachment_codec:
    #    encoding_decision = True
    # Note, attachments may not have a codec name if the attachment is an image
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:t copy'
    return encoding_decision, ffmpeg_command







In [35]:
check_codecs(run_ffprobe('\media/Media 1/test.mkv'))

{
  "programs": [],
  "stream_groups": [],
  "streams": [
    {
      "index": 0,
      "codec_name": "hevc",
      "codec_type": "video"
    },
    {
      "index": 1,
      "codec_name": "aac",
      "codec_type": "audio",
      "channel_layout": "stereo"
    },
    {
      "index": 2,
      "codec_name": "subrip",
      "codec_type": "subtitle"
    }
  ],
  "format": {
    "filename": "\\media/Media 1/test.mkv",
    "nb_streams": 3,
    "nb_programs": 0,
    "nb_stream_groups": 0,
    "format_name": "matroska,webm",
    "format_long_name": "Matroska / WebM",
    "start_time": "0.000000",
    "duration": "452.952000",
    "size": "395300778",
    "bit_rate": "6981768",
    "probe_score": 100,
    "tags": {
      "encoder": "libebml v1.4.4 + libmatroska v1.7.1",
      "creation_time": "2023-09-07T11:39:24.000000Z"
    }
  }
}
There are : 3 streams
Stream 0 is video
Steam 0 codec is: hevc
Steam 1 codec is: aac
audio stream
Steam 2 codec is: subrip
subtitle stream
True
 -map 0:0 -c:v li